# Compare OSS Decision Models

## 개요

대화형 생성(Text Generation)을 배제하고 오직 구조화된 결정(Typed Decision)과 보정된 확률 점수만을 고속 산출하는 오픈소스 의사결정 모델(Decision Model)들의 성능과 추론시간을 비교하는 Jupyter Notebook 벤치마킹 환경입니다.

본 환경에서는 **Gemma 4의 함수 호출(Function Calling / Tool Calling) 기능**을 활용하여 외부 연준 웹페이지(`https://www.federalreserve.gov/monetarypolicy/fomcminutes20260729.htm`)로부터 의사록 원문을 동적으로 수집(`urlfetch`)하고, 이를 정형 상태 JSON으로 압축한 뒤 대표적 오픈소스 결정 모델군(OpenJev, Nimble, Laya)에 주입하여 판정(UP/DOWN), 신뢰도, 추론 지연 시간을 측정합니다.

## Tech Stack Layers 및 파이프라인

```
[데이터 수집 및 전처리 계층 (Data Ingestion & Preprocessing)]
 ──> Gemma 4 Tool Calling: urlfetch 도구 호출 판단 및 실행
 ──> 동적 URL 페칭: 연준 공식 사이트 의사록 HTML 파싱 및 텍스트 정제
 ──> Gemma 4 상태 압축: 장문 텍스트(약 4만 자) -> 512토큰 State JSON 압축

[결정 추론 계층 (Decision Inference)]
 ──> 1. OpenJev (27B Q4_K_M): 상용 Jev 1:1 재현 규격
 ──> 2. Bespoke Nimble (9B Q4_K_M): Qwen 기반 지시 결정 모델
 ──> 3. Laya (0.4B ModernBERT): 초고속 인코더 결정 모델
```

| 구분 | 도구 호출 및 요약기 | 결정 모델 1 | 결정 모델 2 | 결정 모델 3 |
| --- | --- | --- | --- | --- |
| 모델 명칭 | gemma-4-E4B-it-GGUF | OpenJev-GGUF | Bespoke-Nimble-9B-v3-GGUF | Laya |
| HF 저장소 | ggml-org/gemma-4-E4B-it-GGUF | ggml-org/OpenJev-GGUF | ggml-org/Bespoke-Nimble-9B-v3-GGUF | answerdotai/ModernBERT-large |
| 파라미터 규모 | 4.5B | 27B | 9B | 0.4B |
| 백본 아키텍처 | Gemma 4 디코더 | Qwen 27B 디코더 | Qwen 9B 디코더 | ModernBERT-large 인코더 |
| 채택 런타임 | GPU (llama.cpp) | GPU (llama.cpp Q4_K_M) | GPU (llama.cpp Q4_K_M) | GPU (PyTorch Transformers) |
| 주요 역할 | urlfetch 도구 호출 및 상태 압축 | 상용 Jev 규격 오픈소스 재현 | 정밀 보정 확률 스코어링 | 10~30ms 수준의 초고속 판정 |


In [1]:
#@title Colab Secret 기반 HF_TOKEN 설정 및 로컬 작업 디렉토리 초기화
import os
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# 1. Colab Secrets에서 HF_TOKEN 연동
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("[+] Colab Secrets에서 HF_TOKEN 연동 완료")
    else:
        print("[*] HF_TOKEN이 비어 있습니다. 공개 모델은 익명으로 다운로드됩니다.")
except Exception as e:
    print(f"[*] Colab Secrets 미사용 또는 로컬 환경: {e}")

# 2. Colab 로컬 초고속 NVMe 임시 디스크 경로 지정 (FUSE 계층 배제)
LOCAL_MODEL_DIR = "/content/local_models"
os.makedirs(LOCAL_MODEL_DIR, exist_ok=True)
print(f"[+] 로컬 고속 저장소 경로: {LOCAL_MODEL_DIR}")

[+] Colab Secrets에서 HF_TOKEN 연동 완료
[+] 로컬 고속 저장소 경로: /content/local_models


In [2]:
#@title CUDA 버전 점검 및 llama-cpp-python 설치
!nvcc --version

# Colab 환경에 최적화된 공식 cu130 휠 설치
!pip uninstall -y llama-cpp-python
!pip install -U llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu130

!pip install -q transformers accelerate pandas

from llama_cpp import llama_supports_gpu_offload
print("[+] llama.cpp GPU 가속 활성화 여부:", llama_supports_gpu_offload())

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0
Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu130
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.8/782.8 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.7 MB/s eta 0:00:00


ggml_cuda_init: found 1 CUDA devices (Total VRAM: 22563 MiB):
  Device 0: NVIDIA L4, compute capability 8.9, VMM: yes, VRAM: 22563 MiB


[+] llama.cpp GPU 가속 활성화 여부: True


In [3]:
#@title Hugging Face 고속 직통 다운로드 함수
import os
import time
from huggingface_hub import hf_hub_download

def get_local_model_path(repo_id: str, filename: str) -> str:
    """
    Hugging Face CDN에서 Colab 로컬 NVMe 스토리지(/content/local_models)로 직통 다운로드:
    1. 로컬에 이미 존재하면 재다운로드 없이 즉시 반환
    2. 로컬에 없으면 Hugging Face 백본 고속 다운로드 수행 (300MB/s+ 급)
    """
    local_path = os.path.join(LOCAL_MODEL_DIR, filename)

    if os.path.exists(local_path):
        size_gb = os.path.getsize(local_path) / (1024**3)
        print(f"[OK] 로컬 NVMe 캐시 확인: {filename} ({size_gb:.2f} GB)")
        return local_path

    print(f"[*] Hugging Face CDN 고속 직통 다운로드 시작: {filename} (저장소: {repo_id})")
    t0 = time.perf_counter()
    downloaded_path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        local_dir=LOCAL_MODEL_DIR,
        local_dir_use_symlinks=False,
        token=os.environ.get("HF_TOKEN")
    )
    duration = time.perf_counter() - t0
    size_gb = os.path.getsize(downloaded_path) / (1024**3)
    speed_mb = (size_gb * 1024) / max(duration, 0.01)
    print(f"[+] 다운로드 완료 ({duration:.1f}초, {size_gb:.2f} GB, 평균 전송속도 {speed_mb:.1f} MB/s)")
    return downloaded_path


In [4]:
#@title 추론 함수 및 VRAM/RAM 회수 유틸리티

import gc
import json
import time
import torch
import numpy as np
from llama_cpp import Llama

def cleanup_memory():
    """VRAM 및 RAM 즉시 강제 회수"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def run_isolated_summary_gguf(repo_id: str, filename: str, prompt: str, max_tokens: int = 256, n_ctx: int = 8192, n_gpu_layers: int = -1):
    """Gemma 4 전용: 로컬 NVMe 경로에서 모델을 로드하여 상태 요약 JSON 생성 (n_ctx=8192)"""
    model_path = get_local_model_path(repo_id=repo_id, filename=filename)

    print(f"[*] 모델 VRAM 적재 시작: {filename}")
    t_load = time.perf_counter()
    llm = Llama(
        model_path=model_path,
        n_gpu_layers=n_gpu_layers,
        n_ctx=n_ctx,
        verbose=False
    )
    load_ms = round((time.perf_counter() - t_load) * 1000, 1)

    print(f"[*] 상태 요약 추론 수행 중...")
    t_infer = time.perf_counter()
    output = llm(prompt, max_tokens=max_tokens, temperature=0.0)
    result_text = output["choices"][0]["text"].strip()
    infer_ms = round((time.perf_counter() - t_infer) * 1000, 1)

    del llm
    cleanup_memory()

    print(f"[+] 로드 지연: {load_ms} ms / 추론 지연: {infer_ms} ms (VRAM 해제 완료)")
    return result_text, infer_ms

def run_isolated_decision_gguf(repo_id: str, filename: str, prompt: str, n_gpu_layers: int = -1, n_ctx: int = 2048):
    """결정 모델 전용: 로컬 NVMe 경로에서 모델을 로드하여 logprobs 기반 정규화 확률(Confidence) 산출"""
    model_path = get_local_model_path(repo_id=repo_id, filename=filename)

    print(f"[*] 결정 모델 VRAM 적재 시작: {filename}")
    t_load = time.perf_counter()
    llm = Llama(
        model_path=model_path,
        n_gpu_layers=n_gpu_layers,
        n_ctx=n_ctx,
        logits_all=True,
        verbose=False
    )
    load_ms = round((time.perf_counter() - t_load) * 1000, 1)

    print(f"[*] 의사결정 추론 수행 중...")
    t_infer = time.perf_counter()
    output = llm(
        prompt,
        max_tokens=2,
        temperature=0.0,
        logprobs=5
    )
    infer_ms = round((time.perf_counter() - t_infer) * 1000, 1)

    choice_data = output["choices"][0]
    raw_decision = choice_data["text"].strip().upper()
    decision = "UP" if "UP" in raw_decision else ("DOWN" if "DOWN" in raw_decision else raw_decision)

    confidence_str = "N/A"
    if choice_data.get("logprobs") and "top_logprobs" in choice_data["logprobs"]:
        top_tokens = choice_data["logprobs"]["top_logprobs"][0]
        up_logprob = top_tokens.get(" UP", top_tokens.get("UP", None))
        down_logprob = top_tokens.get(" DOWN", top_tokens.get("DOWN", None))

        if up_logprob is not None and down_logprob is not None:
            p_up = np.exp(up_logprob)
            p_down = np.exp(down_logprob)
            prob_up = p_up / (p_up + p_down)
            conf_val = prob_up if decision == "UP" else (1.0 - prob_up)
            confidence_str = f"{conf_val * 100:.2f}%"
        elif up_logprob is not None or down_logprob is not None:
            target_logprob = up_logprob if decision == "UP" else down_logprob
            conf_val = float(np.exp(target_logprob))
            confidence_str = f"{conf_val * 100:.2f}%"

    del llm
    cleanup_memory()

    print(f"[+] 로드 지연: {load_ms} ms / 추론 지연: {infer_ms} ms (VRAM 해제 완료)")
    return decision, confidence_str, infer_ms


In [5]:
#@title 입력 데이터 수집: Gemma 4 함수 호출(Function Calling) & urlfetch 실시간 수집

import re
import urllib.request
import json
import time

# -------------------------------------------------------------------------
# 1. urlfetch 파이썬 도구 함수 정의 (HTML 본문 파싱 및 텍스트 정제)
# -------------------------------------------------------------------------
def urlfetch(url: str) -> str:
    """연준 웹페이지에 HTTP GET 요청을 보내 본문 텍스트를 파싱 및 정제"""
    print(f"[*] urlfetch 실행: {url}")
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    req = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(req, timeout=15) as resp:
        html = resp.read().decode('utf-8', errors='ignore')

    # 연준 의사록 페이지 본문 영역(col-xs-12) 추출
    chunks = re.findall(r'<div[^>]*class=["'][^"']*col-xs-12[^"']*["'][^>]*>(.*?)</div>', html, re.DOTALL)
    if chunks:
        main_chunk = max(chunks, key=len)
    else:
        main_chunk = html

    # HTML 태그 제거 및 공백 정규화
    clean_text = re.sub(r'<[^>]+>', ' ', main_chunk)
    clean_text = re.sub(r'\s+', ' ', clean_text).strip()
    return clean_text

# -------------------------------------------------------------------------
# 2. Gemma 4 함수 호출(Function Calling) 도구 스키마 및 프롬프트 구성
# -------------------------------------------------------------------------
target_fomc_url = "https://www.federalreserve.gov/monetarypolicy/fomcminutes20260729.htm"

tools_declaration = """Available Tools:
- urlfetch(url: str): Fetch and extract clean main text from a web page URL.
"""

tool_call_prompt = f"""<start_of_turn>user
You are an autonomous economic analyst assistant with tool calling capabilities.
{tools_declaration}

Task: We need the FOMC meeting minutes to evaluate market decisions.
Fetch the document from the following URL using the available tool:
{target_fomc_url}

Respond ONLY with the function call in JSON or call format, for example:
call:urlfetch{{url: "{target_fomc_url}"}}<end_of_turn>
<start_of_turn>model
"""

print("--- [Step 0/5] Gemma 4 도구 호출 트리거 ---")

# Gemma 4 로컬 모델을 통해 함수 호출 생성
tool_output_str, tool_call_lat = run_isolated_summary_gguf(
    repo_id="ggml-org/gemma-4-E4B-it-GGUF",
    filename="gemma-4-E4B-it-Q4_0.gguf",
    prompt=tool_call_prompt,
    max_tokens=64,
    n_ctx=2048,
    n_gpu_layers=-1
)

print(f"[+] Gemma 4 모델 출력: {tool_output_str.strip()} (지연: {tool_call_lat} ms)")

# -------------------------------------------------------------------------
# 3. 모델의 함수 호출 해석 및 실제 urlfetch 실행
# -------------------------------------------------------------------------
url_to_fetch = target_fomc_url
# 모델 출력에서 URL 추출 시도 (정규식 또는 JSON 파싱)
matched_url = re.search(r'https?://[^"'\s\}]+', tool_output_str)
if matched_url:
    url_to_fetch = matched_url.group(0)

print(f"[*] 도구 실행 대상 URL: {url_to_fetch}")
t_start = time.perf_counter()
try:
    raw_fed_statement = urlfetch(url_to_fetch)
    fetch_sec = round(time.perf_counter() - t_start, 2)
    print(f"[+] 의사록 원문 실시간 수집 성공! ({len(raw_fed_statement):,}자, 소요시간: {fetch_sec}초)")
    print(f"[*] 본문 미리보기: {raw_fed_statement[:250]}...")
except Exception as e:
    print(f"[!] urlfetch 요청 실패: {e}")
    raise e


In [6]:
#@title 실행: Step 1. 요약 모델 (Gemma 4 상태 압축)

# -------------------------------------------------------------------------
# Step 1. Gemma 4 기반 상태 압축 (n_ctx=8192 활용)
# -------------------------------------------------------------------------
summary_prompt = f"""user
You are an economic research assistant.
Read the Federal Reserve statement below and extract a structured state for market prediction.

Federal Reserve Statement:
{raw_fed_statement[:15000]}

Extract the key facts into the following exact JSON format.
In "summary_facts", provide a 2-3 sentence dense factual summary of the statement.
DO NOT change or truncate the "choices" array.

Required JSON Structure:
{{
  "macro_tone": "hawkish",
  "summary_facts": "The FOMC kept rates unchanged at 4.25-4.50%. Economic growth and job gains remain solid, but inflation remains elevated above the 2% target. Rate cuts are on hold until greater confidence is achieved.",
  "question": "Will NDX close higher next week?",
  "choices": ["UP", "DOWN"]
}}

Output JSON only:
model
"""

print("\n--- [Step 1/4] Gemma 4 요약 및 JSON 직렬화 ---")

state_json_str, gemma_lat = run_isolated_summary_gguf(
    repo_id="ggml-org/gemma-4-E4B-it-GGUF",
    filename="gemma-4-E4B-it-Q4_0.gguf",
    prompt=summary_prompt,
    max_tokens=256,
    n_ctx=8192,
    n_gpu_layers=-1
)

print("\n[압축된 상태 JSON]\n", state_json_str)

# -------------------------------------------------------------------------
# Step 2. 결정 모델 공통 입력 프롬프트 생성
# -------------------------------------------------------------------------
try:
    clean_json_str = state_json_str.replace("```json", "").replace("```", "").strip()
    state_data = json.loads(clean_json_str)
    summary_text = state_data.get("summary_facts", clean_json_str)
except Exception:
    summary_text = state_json_str

decision_prompt = f"""[System One Decision Task]
Context: {summary_text}
Question: Will NDX close higher next week?
Allowed Choices: [UP, DOWN]
Decision:"""

print("\n[생성된 결정 모델 공통 프롬프트]\n", decision_prompt)



--- [Step 1/4] Gemma 4 요약 및 JSON 직렬화 ---
[*] Hugging Face CDN 고속 직통 다운로드 시작: gemma-4-E4B-it-Q4_0.gguf (저장소: ggml-org/gemma-4-E4B-it-GGUF)


gemma-4-E4B-it-Q4_0.gguf: reconstructing file:   0%|          |  0.00B / 4.59GB            

gemma-4-E4B-it-Q4_0.gguf: downloading bytes:           |  0.00B            

[+] 다운로드 완료 (15.7초, 4.28 GB, 평균 전송속도 279.5 MB/s)
[*] 모델 VRAM 적재 시작: gemma-4-E4B-it-Q4_0.gguf


llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 1024
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 1024


[*] 상태 요약 추론 수행 중...
[+] 로드 지연: 2365.9 ms / 추론 지연: 4668.8 ms (VRAM 해제 완료)

[압축된 상태 JSON]
 ```json
{
  "macro_tone": "neutral",
  "summary_facts": "The FOMC maintained the federal funds rate target range at 3-1/2 to 3-3/4 percent, citing solid economic growth and stable labor markets. While inflation remains elevated due to supply shocks, the Committee signaled continued resolve to achieve price stability. Market expectations for future hikes remain high, with some participants favoring a 25 basis point increase.",
  "question": "Will NDX close higher next week?",
  "choices": ["UP", "DOWN"]
}
```

[생성된 결정 모델 공통 프롬프트]
 [System One Decision Task]
Context: The FOMC maintained the federal funds rate target range at 3-1/2 to 3-3/4 percent, citing solid economic growth and stable labor markets. While inflation remains elevated due to supply shocks, the Committee signaled continued resolve to achieve price stability. Market expectations for future hikes remain high, with some participants fav

In [7]:
#@title 실행: Step 2-1. OpenJev (27B) 추론

# -------------------------------------------------------------------------
# Step 2-1. OpenJev (27B: T4 VRAM 감안하여 20개 레이어 GPU 가속, 나머지 RAM 오프로드)
# -------------------------------------------------------------------------
print("\n--- [Step 2/4] OpenJev (27B) 추론 ---")
try:
    openjev_dec, openjev_conf, openjev_lat = run_isolated_decision_gguf(
        repo_id="ggml-org/OpenJev-GGUF",
        filename="OpenJev-Q4_K_M.gguf",
        prompt=decision_prompt,
        n_gpu_layers=20
    )
    print(f"\n[OpenJev 결과] 결정: {openjev_dec} | 신뢰도: {openjev_conf} | 지연시간: {openjev_lat} ms")
except Exception as e:
    print(f"[!] OpenJev 추론 실패: {e}")
    openjev_dec, openjev_conf, openjev_lat = "Error", "N/A", "N/A"



--- [Step 2/4] OpenJev (27B) 추론 ---
[*] Hugging Face CDN 고속 직통 다운로드 시작: OpenJev-Q4_K_M.gguf (저장소: ggml-org/OpenJev-GGUF)


OpenJev-Q4_K_M.gguf: reconstructing file:   0%|          |  0.00B / 19.0GB            

OpenJev-Q4_K_M.gguf: downloading bytes:           |  0.00B            

[+] 다운로드 완료 (51.2초, 17.67 GB, 평균 전송속도 353.4 MB/s)
[*] 결정 모델 VRAM 적재 시작: OpenJev-Q4_K_M.gguf
[*] 의사결정 추론 수행 중...
[+] 로드 지연: 3479.2 ms / 추론 지연: 5000.6 ms (VRAM 해제 완료)

[OpenJev 결과] 결정: DOWN | 신뢰도: 55.80% | 지연시간: 5000.6 ms


In [8]:
#@title 실행: Step 2-2. Bespoke Nimble (9B) 추론

# -------------------------------------------------------------------------
# Step 2-2. Bespoke Nimble (9B: VRAM 6~7GB 점유, 전 레이어 GPU 가속)
# -------------------------------------------------------------------------
print("\n--- [Step 3/4] Bespoke Nimble (9B) 추론 ---")
try:
    nimble_dec, nimble_conf, nimble_lat = run_isolated_decision_gguf(
        repo_id="ggml-org/Bespoke-Nimble-9B-v3-GGUF",
        filename="Bespoke-Nimble-9B-v3-Q4_K_M.gguf",
        prompt=decision_prompt,
        n_gpu_layers=-1
    )
    print(f"\n[Nimble 결과] 결정: {nimble_dec} | 신뢰도: {nimble_conf} | 지연시간: {nimble_lat} ms")
except Exception as e:
    print(f"[!] Nimble 추론 실패: {e}")
    nimble_dec, nimble_conf, nimble_lat = "Error", "N/A", "N/A"



--- [Step 3/4] Bespoke Nimble (9B) 추론 ---
[*] Hugging Face CDN 고속 직통 다운로드 시작: Bespoke-Nimble-9B-v3-Q4_K_M.gguf (저장소: ggml-org/Bespoke-Nimble-9B-v3-GGUF)


Bespoke-Nimble-9B-v3-Q4_K_M.gguf: reconstructing file:   0%|          |  0.00B / 6.32GB            

Bespoke-Nimble-9B-v3-Q4_K_M.gguf: downloading bytes:           |  0.00B            

[+] 다운로드 완료 (18.5초, 5.89 GB, 평균 전송속도 326.7 MB/s)
[*] 결정 모델 VRAM 적재 시작: Bespoke-Nimble-9B-v3-Q4_K_M.gguf
[*] 의사결정 추론 수행 중...
[+] 로드 지연: 2692.1 ms / 추론 지연: 1371.6 ms (VRAM 해제 완료)

[Nimble 결과] 결정: UP | 신뢰도: 81.81% | 지연시간: 1371.6 ms


In [14]:
#@title 실행: Step 2-3. Laya (0.4B) 추론 (ModernBERT 인코더 GPU 가속)
import time
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("\n--- [Step 4/4] Laya (0.4B) 추론 (Transformers GPU) ---")

try:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[*] 연산 가속 장치: {device}")

    # 1. 모델 및 토크나이저 초기화 (VRAM 적재)
    print("[*] Laya (ModernBERT-large) 로드 중...")
    model_name = "answerdotai/ModernBERT-large"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)
    model.eval()

    # Warm-up 패스를 통한 GPU 초기화 오버헤드 제거
    _ = model(**{k: v.to(device) for k, v in tokenizer("warmup", return_tensors="pt").items()})

    # 2. 추론 지연시간 측정
    print("[*] 의사결정 추론 시작...")
    t_start = time.perf_counter()

    inputs = tokenizer(decision_prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=-1)

    pred_class_idx = torch.argmax(probabilities, dim=-1).item()
    laya_dec = "UP" if pred_class_idx == 1 else "DOWN"
    laya_conf = f"{probabilities[0][pred_class_idx].item() * 100:.2f}%"
    laya_lat = round((time.perf_counter() - t_start) * 1000, 1)

    # 3. 자원 반환 및 메모리 회수
    del model, tokenizer
    cleanup_memory()

    print(f"\n[Laya 결과] 결정: {laya_dec} | 신뢰도: {laya_conf} | 순수 추론 지연시간: {laya_lat} ms (VRAM 해제 완료)")

except Exception as e:
    print(f"[!] Laya 추론 중 예외 발생: {e}")
    laya_dec, laya_conf, laya_lat = "Error", "N/A", "N/A"


--- [Step 4/4] Laya (0.4B) 추론 (Transformers GPU) ---
[*] 연산 가속 장치: cuda
[*] Laya (ModernBERT-large) 로드 중...


Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-large
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[*] 의사결정 추론 시작...

[Laya 결과] 결정: DOWN | 신뢰도: 78.51% | 순수 추론 지연시간: 28.9 ms (VRAM 해제 완료)


In [15]:
#@title 출력 결과 비교 테이블
# -------------------------------------------------------------------------
import pandas as pd

df_comparison = pd.DataFrame([
    {"Model": "Gemma 4 (Compressor)", "Params": "4.5B", "Device": "GPU", "Decision": "State Compressed", "Confidence": "N/A", "Latency": f"{gemma_lat} ms" if 'gemma_lat' in globals() else "N/A"},
    {"Model": "OpenJev", "Params": "27B", "Device": "GPU", "Decision": openjev_dec if 'openjev_dec' in globals() else "N/A", "Confidence": openjev_conf if 'openjev_conf' in globals() else "N/A", "Latency": f"{openjev_lat} ms" if 'openjev_lat' in globals() else "N/A"},
    {"Model": "Nimble v3", "Params": "9B", "Device": "GPU", "Decision": nimble_dec if 'nimble_dec' in globals() else "N/A", "Confidence": nimble_conf if 'nimble_conf' in globals() else "N/A", "Latency": f"{nimble_lat} ms" if 'nimble_lat' in globals() else "N/A"},
    {"Model": "Laya", "Params": "0.4B", "Device": "GPU", "Decision": laya_dec if 'laya_dec' in globals() else "N/A", "Confidence": laya_conf if 'laya_conf' in globals() else "N/A", "Latency": f"{laya_lat} ms" if 'laya_lat' in globals() else "N/A"}
])

display(df_comparison)

,Model,Params,Device,Decision,Confidence,Latency
0,Gemma 4 (Compressor),4.5B,GPU,State Compressed,N/A,4668.8 ms
1,OpenJev,27B,GPU,DOWN,55.80%,5000.6 ms
2,Nimble v3,9B,GPU,UP,81.81%,1371.6 ms
3,Laya,0.4B,GPU,DOWN,78.51%,28.9 ms
